# Índice + muestra de 50 preguntas en Kaggle (GPU) + evaluador oficial

Mismo flujo que `muestra_en_colab.ipynb`, para cuando Colab no da GPU: (1) actualiza el índice en
la GPU (con la caché, solo los fragmentos nuevos), (2) responde `data/sample_50.jsonl` con el
sistema completo (bge-m3 + reranker + decoder de la celda 4, temperatura 0) y (3) califica con
`scripts/evaluate.py`, sin juez y con RAGAS. Devuelve un único zip con índice + resultados.

**Preparación (una sola vez):**
1. Cuenta de Kaggle **verificada con teléfono** (*Settings → Phone verification*): sin eso no hay
   GPU ni internet.
2. En el computador: `python -m src.index.paquete_colab --modo muestra` →
   `dist/paquete_colab_muestra.zip`.
3. Kaggle → *Datasets → New Dataset* → subir ese zip, título **`paquete-colab-muestra`**, privado →
   *Create*. Kaggle puede descomprimirlo solo: el cuaderno funciona en ambos casos.
4. Kaggle → *Code → New Notebook* → *File → Import Notebook* → este archivo.
5. En el panel derecho del cuaderno:
   - *Session options → Accelerator* → **GPU T4 x2** (no P100: llama.cpp precompilado no la soporta).
   - *Session options → Internet* → **On** (descarga modelos de Hugging Face y dependencias).
   - *Input → Add Input* → *Your Work → Datasets* → `paquete-colab-muestra`.
6. **Llave del juez:** menú *Add-ons → Secrets* → *Add secret* → nombre `OPENROUTER_API_KEY`, valor =
   la llave → marcar la casilla para adjuntarlo a este cuaderno.

Luego *Run All*. Kaggle da ~30 h de GPU por semana y sesiones de hasta 12 h.


In [ ]:
# 1. Comprobar GPU (deben aparecer las T4)
!nvidia-smi --query-gpu=name,memory.total --format=csv


## 2. Traer el paquete desde el Dataset de Kaggle


In [ ]:
import glob, os
TAG = "muestra_v10"   # carpeta de resultados: runs/<TAG> (no pisa corridas anteriores)
# Preguntas a responder: "" = las 50 (corrida completa, con evaluación aquí mismo).
# Con una lista de ids (experimento parcial) se responden solo esas y se evalúan en el computador
# con `python -m src.eval.combinar` (p. ej. las 15 MC: "51,58,60,128,290,308,352,358,487,528,600,617,647,671,748").
IDS = ""
# Rango por posición en el archivo (no por id), ambos extremos incluidos: "1 15" = de la 1.ª a la
# 15.ª. Sirve para repartir entre máquinas (el sábado: "1 250", "251 500"…). "" = sin rango.
RANGO = ""
PARCIAL = bool(IDS or RANGO)
ARG_IDS = (f"--ids {IDS} " if IDS else "") + (f"--rango {RANGO}" if RANGO else "")

# El Dataset aparece en /kaggle/input/<nombre>/ (solo lectura), como zip o ya descomprimido.
# Si hay varios paquetes en Input, se usa el más reciente según su PAQUETE.txt (fecha de
# generación); los paquetes viejos sin PAQUETE.txt quedan al final. Lo ideal: uno solo en Input.
def _fecha(raiz):
    f = os.path.join(raiz, "PAQUETE.txt")
    return open(f, encoding="utf-8").readline().strip() if os.path.isfile(f) else ""

!rm -rf /kaggle/working/proyecto && mkdir -p /kaggle/working/proyecto
raices = [os.path.dirname(os.path.dirname(p))
          for p in glob.glob("/kaggle/input/**/data/sample_50.jsonl", recursive=True)]
zips = glob.glob("/kaggle/input/**/paquete_colab_muestra*.zip", recursive=True)
if raices:
    raices.sort(key=_fecha, reverse=True)
    for r in raices:
        print("Paquete en Input:", r, "→", _fecha(r) or "sin PAQUETE.txt (viejo)")
    print("Se usa:", raices[0])
    !cp -r "{raices[0]}"/. /kaggle/working/proyecto/
elif zips:
    print("Zip encontrado:", zips[0])
    !unzip -q "{zips[0]}" -d /kaggle/working/proyecto
else:
    raise SystemExit("No encuentro el paquete: agregue el Dataset con paquete_colab_muestra.zip en Input (panel derecho).")
%cd /kaggle/working/proyecto
!ls -la build/indice build/cache/emb data
!cat PAQUETE.txt   # debe coincidir con lo que imprimió paquete_colab al generar el zip


## 3. Instalar dependencias (≈ 3–5 min)

Se conserva el `torch` con CUDA de Kaggle. `llama-cpp-python` se instala con soporte CUDA para
correr Qwen3-8B (GGUF de 4 bits) en la GPU. Si no hay rueda precompilada para esta versión de
Python, la celda 3b la compila (≈ 15 min).

In [ ]:
!pip install -q sentence-transformers==6.1.0 transformers==5.17.0 faiss-cpu==1.15.1 bm25s==0.3.11 PyStemmer==3.1.0 python-dotenv==1.2.3 jsonschema==4.26.0 httpx huggingface_hub
!pip install -q llama-cpp-python==0.3.35 --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124 --only-binary llama-cpp-python || echo "SIN RUEDA PRECOMPILADA: ejecute la celda 3b"

In [ ]:
# 3b. SOLO si la celda anterior dijo "SIN RUEDA PRECOMPILADA" (compila con CUDA, ≈ 15 min)
!CMAKE_ARGS="-DGGML_CUDA=on" pip install -q llama-cpp-python==0.3.35 --no-binary llama-cpp-python --force-reinstall --no-deps

In [ ]:
# 3c. Verificar que llama.cpp ve la GPU (debe decir True).
# La rueda precompilada de llama.cpp es para CUDA 12. Si la imagen del entorno trae otra versión
# (p. ej. Colab con CUDA 13: "libcudart.so.12: cannot open shared object file"), se instalan las
# librerías de CUDA 12 como paquetes de Python y se apunta a ellas; los !python heredan la ruta.
import ctypes, glob, os, subprocess, sys
prueba = subprocess.run([sys.executable, "-c", "import llama_cpp"], capture_output=True, text=True)
if prueba.returncode != 0 and ".so.12" in prueba.stderr:
    print("Faltan librerías de CUDA 12: instalándolas…")
    !pip install -q nvidia-cuda-runtime-cu12 nvidia-cublas-cu12
    libs = [d for pat in ("nvidia/cuda_runtime/lib", "nvidia/cublas/lib")
            for d in glob.glob(f"/usr/local/lib/python3*/dist-packages/{pat}")]
    os.environ["LD_LIBRARY_PATH"] = ":".join(libs + [os.environ.get("LD_LIBRARY_PATH", "")])
    for d in libs:
        for so in sorted(glob.glob(d + "/lib*.so.12")):
            ctypes.CDLL(so, mode=ctypes.RTLD_GLOBAL)
import llama_cpp
print("GPU disponible para llama.cpp:", llama_cpp.llama_supports_gpu_offload())


## 4. Configurar todo en GPU y actualizar el índice

Con la caché de embeddings solo se recalculan los fragmentos nuevos o modificados. Debe terminar
con `CÓDIGO DE SALIDA: 0`.


In [ ]:
# por si otra celda cambió de carpeta (permite reejecutar en cualquier orden)
%cd /kaggle/working/proyecto
import os
# Decoder: por defecto Gemma 4 E4B en GGUF Q8_0 (C-15). Para Qwen3-8B: "Qwen/Qwen3-8B-GGUF"
# y "Qwen3-8B-Q4_K_M.gguf". Para probar otro modelo abierto (≤ 8B):
# agregarlo a MODELOS_ABIERTOS en src/config.py con su licencia y cambiar estas dos líneas
# (repositorio y archivo .gguf de Hugging Face). Ver docs/COMO_EJECUTAR.md §10.
DECODER_GGUF_REPO = "ggml-org/gemma-4-E4B-it-GGUF"
DECODER_GGUF_FILE = "gemma-4-E4B-it-Q8_0.gguf"
os.environ.update({"DECODER_GGUF_REPO": DECODER_GGUF_REPO, "DECODER_GGUF_FILE": DECODER_GGUF_FILE})
os.environ.update({"ENCODER_DEVICE": "cuda", "ENCODER_BATCH_SIZE": "64",
                   "DECODER_DEVICE": "cuda", "DECODER_BACKEND": "llamacpp",
                   "CORPUS_SOURCE": "local", "PYTHONIOENCODING": "utf-8",
                   # Kaggle trae dos T4: se usa una, como en Colab, para que los tiempos sean comparables.
                   "CUDA_VISIBLE_DEVICES": "0",
                   # JAX/TF preinstalados reservan el 75 % de la VRAM si se importan: a CPU.
                   "JAX_PLATFORMS": "cpu", "XLA_PYTHON_CLIENT_PREALLOCATE": "false",
                   "TF_FORCE_GPU_ALLOW_GROWTH": "true", "USE_TF": "0", "USE_JAX": "0"})
!python -m src.index.build; echo "CÓDIGO DE SALIDA: $? (0 = bien)"
!cat build/indice/index_manifest.json


In [ ]:
# 4b. Probar el decoder en la GPU
# por si otra celda cambió de carpeta (permite reejecutar en cualquier orden)
%cd /kaggle/working/proyecto
!python -m src.generation.ping --sin-cache
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv


## 5. Responder las 50 preguntas de la muestra

Escribe cada respuesta al terminarla en `runs/<TAG>/`. Si Colab se desconecta, vuelva a
ejecutar las celdas 2 y 4 (el índice ya calculado se pierde con la sesión, pero la caché lo
rehace rápido) y luego **esta celda**: continúa donde quedó si `runs/` sigue ahí.


In [ ]:
# por si otra celda cambió de carpeta (permite reejecutar en cualquier orden)
%cd /kaggle/working/proyecto
!python -m src.pipeline.main --split sample --tag {TAG} {ARG_IDS}; echo "CÓDIGO DE SALIDA: $? (0 = bien; cualquier otro número = error, revise arriba)"
!cat runs/{TAG}/tiempos.json


## 6. Evaluador oficial sin juez (50 puntos deterministas)

In [ ]:
# por si otra celda cambió de carpeta (permite reejecutar en cualquier orden)
%cd /kaggle/working/proyecto
if PARCIAL:
    # Corrida parcial: el evaluador oficial necesita las 50; aquí solo la exactitud de selección
    # múltiple sobre lo respondido. La evaluación completa se hace en el computador.
    import json
    oro = {q["id"]: q for q in map(json.loads, open("data/sample_50.jsonl", encoding="utf-8"))}
    resp = [json.loads(l) for l in open(f"runs/{TAG}/submissions.jsonl", encoding="utf-8")]
    trazas = {t["id"]: t for t in map(json.loads, open(f"runs/{TAG}/trazas.jsonl", encoding="utf-8"))}
    mc = [r for r in resp if oro[r["id"]]["formato"] == "multiple_choice"]
    bien = 0
    for r in mc:
        q = oro[r["id"]]
        ok = r.get("respuesta_correcta") == q["respuesta_correcta"]
        bien += ok
        sim = trazas.get(r["id"], {}).get("opcion_por_similitud")
        print(f'{r["id"]:>4}  correcta {q["respuesta_correcta"]}  modelo {r.get("respuesta_correcta")} '
              f'{"✓" if ok else "✗"}  (por similitud: {sim})')
    print(f"\nSelección múltiple: {bien} de {len(mc)}. Resto de la evaluación: en el computador.")
else:
    !python scripts/evaluate.py --submission runs/{TAG}/submissions.jsonl --split sample --out runs/{TAG}/reporte.json


## 7. Cargar la llave del juez

Toma `OPENROUTER_API_KEY` de *Add-ons → Secrets* de Kaggle. Si no está, la pide sin mostrarla.
Nunca se imprime ni se guarda en archivos.


In [ ]:
import os, getpass
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["OPENROUTER_API_KEY"] = UserSecretsClient().get_secret("OPENROUTER_API_KEY")
except Exception:
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OPENROUTER_API_KEY: ")
print("Llave cargada:", bool(os.environ.get("OPENROUTER_API_KEY")), "(no se muestra)")


## 8. Evaluador oficial con el juez RAGAS (80 puntos automáticos)

In [ ]:
# por si otra celda cambió de carpeta (permite reejecutar en cualquier orden)
%cd /kaggle/working/proyecto
if PARCIAL:
    print("Corrida parcial (IDS o RANGO): sin RAGAS; salte a la celda 9.")
else:
    !pip install -q -r scripts/requirements-evaluador.txt
    !python scripts/evaluate.py --submission runs/{TAG}/submissions.jsonl --split sample --ragas --out runs/{TAG}/reporte_ragas.json


## 9. Guardar índice + resultados

Un solo zip con `runs/<TAG>/`, el índice nuevo (`index.faiss`, `index_manifest.json`, `bm25/`) y
la caché de embeddings actualizada, en `/kaggle/working/`. Para descargarlo: panel derecho →
*Output* (o *Data → Output*) → `/kaggle/working/<TAG>.zip` → menú ⋮ → *Download*. En el
computador, dejarlo en la raíz del proyecto.


In [ ]:
!cd /kaggle/working/proyecto && zip -q -r /kaggle/working/{TAG}.zip runs/{TAG} build/indice/index.faiss build/indice/index_manifest.json build/indice/bm25 build/cache/emb
!ls -la /kaggle/working/{TAG}.zip
%cd /kaggle/working
from IPython.display import FileLink
FileLink(f"{TAG}.zip")   # enlace de descarga (también en el panel Output)
